In [ ]:
import requests, json, time
from google.colab import drive, userdata
from datasets import load_dataset
from collections import Counter

drive.mount('/content/drive')
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

r = requests.get(
    "https://api.together.xyz/v1/models",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}"},
    timeout=20
)
models = r.json()
ids = [m['id'] for m in models]

print("=== Candidate models ===")
for needle in ['mistral', 'gemma', 'phi-3', 'phi_3', 'Mistral', 'Gemma', 'Phi']:
    matches = [m for m in ids if needle.lower() in m.lower()][:5]
    if matches:
        print(f"\n  '{needle}' matches:")
        for m in matches:
            print(f"    {m}")

In [ ]:
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")

MODELS = {
    "gemma":   "google/gemma-2-9b-it",
    "mistral": "nim/nv-mistralai/mistral-nemo-12b-instruct",
}

def get_answer(question, options, api_key, model):

    prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""

    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 5,
                "temperature": 0.0
            },
            timeout=30
        )
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}: {r.text[:150]}"
        text = r.json()['choices'][0]['message']['content'].strip().upper()

        for char in text:
            if char in 'ABCD':
                return char, None
        return None, f"No letter in response: {text!r}"
    except Exception as e:
        return None, str(e)

for name, model_id in MODELS.items():
    print(f"\n=== {name.upper()} ({model_id}) ===")
    results = []
    consecutive_errors = 0

    for i, row in enumerate(dataset['test']):
        pred, err = get_answer(row['question'], row['options'], TOGETHER_API_KEY, model_id)

        if pred is None:
            consecutive_errors += 1
            if consecutive_errors <= 3:
                print(f"  [{i}] error: {err}")
            if consecutive_errors >= 10:
                print(f"  >>> 10 consecutive errors, aborting {name} <<<")
                break
        else:
            consecutive_errors = 0

        results.append({
            'idx': i,
            'gold': row['answer_idx'],
            'pred': pred,
            'correct': int(pred == row['answer_idx']) if pred else 0
        })

        if (i + 1) % 200 == 0:
            valid = sum(1 for r in results if r['pred'] is not None)
            acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
            print(f"  {i+1}/{len(dataset['test'])} — running acc: {acc_so_far:.1f}% ({valid} valid)")

        time.sleep(0.1)

    total = len(results)
    valid = [r for r in results if r['pred'] is not None]
    correct = sum(r['correct'] for r in results)
    print(f"\n  Total: {total}")
    print(f"  Valid responses: {len(valid)}")
    print(f"  Accuracy: {correct}/{total} = {correct/total*100:.1f}%")

    with open(f'/content/drive/MyDrive/{name}_results.json', 'w') as f:
        json.dump(results, f)
    print(f"  Saved to /content/drive/MyDrive/{name}_results.json")

print("\n=== All evaluations complete ===")
print("Existing accuracies for comparison:")
print("  Flan-t5-base:    26.5%")
print("  Llama-3-8B-Lite: 52.0%")
print("  Qwen-2.5-7B:     59.5%")

In [ ]:
candidates = [

    "mistralai/Mistral-Small-24B-Instruct-2501",
    "mistralai/Magistral-Small-2506",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",

    "google/gemma-3n-E4B-it",
    "google/gemma-4-E4B-it",
    "google/gemma-4-26B-A4B-it",

    "NousResearch/Nous-Hermes-2-Mixtral-8x7B-DPO",

    "deepseek-ai/DeepSeek-V3",
    "deepseek-ai/deepseek-llm-67b-chat",

    "zero-one-ai/Yi-34B-Chat",

    "meta-llama/Llama-Guard-4-12B",
]

print("Testing each candidate with a ping call...\n")
working = []
for model in candidates:
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
            json={
                "model": model,
                "messages": [{"role": "user", "content": "Reply with only: PING"}],
                "max_tokens": 5,
                "temperature": 0.0
            },
            timeout=15
        )
        if r.status_code == 200:
            content = r.json()['choices'][0]['message']['content'].strip()
            print(f"  ✓ {model}")
            print(f"     → {content!r}")
            working.append(model)
        else:
            err = r.json().get('error', {}).get('message', r.text[:100])
            print(f"  ✗ {model}")
            print(f"     {err[:120]}")
    except Exception as e:
        print(f"  ✗ {model}  ({e})")
    time.sleep(0.2)

print(f"\n=== {len(working)} serverless models found ===")
for m in working:
    print(f"  {m}")

In [ ]:
MODELS = {
    "gemma3n":  "google/gemma-3n-E4B-it",
    "deepseek": "deepseek-ai/DeepSeek-V3",
}

def get_answer(question, options, api_key, model):
    prompt = f"""Answer this USMLE medical question. Reply with only A, B, C, or D.

Question: {question}

A: {options['A']}
B: {options['B']}
C: {options['C']}
D: {options['D']}

Answer:"""
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={
                "model": model,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": 10,
                "temperature": 0.0
            },
            timeout=45
        )
        if r.status_code != 200:
            return None, f"HTTP {r.status_code}: {r.text[:150]}"
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for char in text:
            if char in 'ABCD':
                return char, None
        return None, f"No letter: {text!r}"
    except Exception as e:
        return None, str(e)

for name, model_id in MODELS.items():
    print(f"\n=== {name.upper()} ({model_id}) ===")
    results = []
    consecutive_errors = 0
    for i, row in enumerate(dataset['test']):
        pred, err = get_answer(row['question'], row['options'], TOGETHER_API_KEY, model_id)
        if pred is None:
            consecutive_errors += 1
            if consecutive_errors <= 3:
                print(f"  [{i}] error: {err}")
            if consecutive_errors >= 10:
                print(f"  >>> 10 consecutive errors, aborting {name} <<<")
                break
        else:
            consecutive_errors = 0
        results.append({
            'idx': i,
            'gold': row['answer_idx'],
            'pred': pred,
            'correct': int(pred == row['answer_idx']) if pred else 0
        })
        if (i + 1) % 200 == 0:
            acc_so_far = sum(r['correct'] for r in results) / len(results) * 100
            print(f"  {i+1}/{len(dataset['test'])} — running acc: {acc_so_far:.1f}%")
        time.sleep(0.15)

    total = len(results)
    correct = sum(r['correct'] for r in results)
    print(f"  Final: {correct}/{total} = {correct/total*100:.1f}%")
    with open(f'/content/drive/MyDrive/{name}_results.json', 'w') as f:
        json.dump(results, f)
    print(f"  Saved to /content/drive/MyDrive/{name}_results.json")

In [ ]:
import json, numpy as np
from collections import Counter
from itertools import combinations

models_data = {}
for name in ['flan_t5', 'llama', 'qwen', 'gemma3n', 'deepseek']:
    with open(f'/content/drive/MyDrive/{name}_results.json') as f:
        results = sorted(json.load(f), key=lambda x: x['idx'])
        models_data[name] = np.array([r['correct'] for r in results])

N = len(models_data['flan_t5'])
print(f"N = {N} questions per model")
for name, arr in models_data.items():
    print(f"  {name:10s}: {arr.mean()*100:.1f}% accuracy")

strong_names = ['llama', 'qwen', 'gemma3n', 'deepseek']
flan = models_data['flan_t5']

print(f"\n{'='*70}")
print("Intersection analysis: Flan correct AND k strong models wrong")
print(f"{'='*70}\n")

B = 10000
rng = np.random.default_rng(seed=42)

def run_perm_test(condition_mask_fn, label):
    """Compute observed intersection count + null distribution via permutation."""
    observed = int((flan == 1).sum() & 0) + int(condition_mask_fn(models_data, flan).sum())
    obs = int(condition_mask_fn(models_data, flan).sum())

    null = np.zeros(B, dtype=int)
    for b in range(B):
        shuffled = {name: rng.permutation(arr) for name, arr in models_data.items()}
        null[b] = int(condition_mask_fn(shuffled, shuffled['flan_t5']).sum())

    nm, ns = null.mean(), null.std()
    lo, hi = np.percentile(null, [2.5, 97.5])
    z = (obs - nm) / ns if ns > 0 else 0
    p = (null >= obs).mean() if obs > nm else 1 - (null <= obs).mean()

    print(f"{label}")
    print(f"  Observed: {obs} | Null mean: {nm:.1f} ± {ns:.1f} | 95% CI: [{lo:.0f}, {hi:.0f}]")
    print(f"  Z = {z:+.2f} | p (one-sided) = {p:.4f}")
    print()
    return obs, nm, ns, z, p

print("--- Flan right AND k of 4 strong models wrong ---\n")
for k in [1, 2, 3, 4]:
    def cond(d, f, k=k):
        wrong_count = sum((d[s] == 0).astype(int) for s in strong_names)
        return (f == 1) & (wrong_count == k)
    run_perm_test(cond, f"k = {k} (Flan right, exactly {k} strong wrong)")

print("--- Flan right AND ALL 4 strong models wrong (strict trap) ---\n")
def all_strong_wrong(d, f):
    cond_all = (f == 1)
    for s in strong_names:
        cond_all = cond_all & (d[s] == 0)
    return cond_all
run_perm_test(all_strong_wrong, "Flan right AND all 4 strong wrong")

print("--- Flan right AND DeepSeek-V3 (frontier) wrong ---\n")
def flan_beats_deepseek(d, f):
    return (f == 1) & (d['deepseek'] == 0)
run_perm_test(flan_beats_deepseek, "Flan right AND DeepSeek wrong (any other model state)")

print("--- DeepSeek wrong AND all 3 mid-tier wrong AND Flan right ---\n")
def cleanest_trap(d, f):
    return (f == 1) & (d['llama'] == 0) & (d['qwen'] == 0) & (d['gemma3n'] == 0) & (d['deepseek'] == 0)
run_perm_test(cleanest_trap, "Cleanest trap: all 4 strong wrong, Flan right")

print(f"\n{'='*70}")
print("Summary: if any z > 4, that's the paper. If all z < 2.5, finalize methodology framing.")
print(f"{'='*70}")

In [ ]:
llama_c = models_data['llama']
qwen_c = models_data['qwen']
gemma_c = models_data['gemma3n']
deep_c = models_data['deepseek']
flan_c = models_data['flan_t5']

all_strong_wrong = (llama_c == 0) & (qwen_c == 0) & (gemma_c == 0) & (deep_c == 0)
all_strong_right = (llama_c == 1) & (qwen_c == 1) & (gemma_c == 1) & (deep_c == 1)
mixed = ~all_strong_wrong & ~all_strong_right

print(f"=== Flan accuracy stratified by strong-model agreement ===\n")

for label, mask in [
    ("ALL 4 strong WRONG", all_strong_wrong),
    ("ALL 4 strong RIGHT", all_strong_right),
    ("MIXED (some right, some wrong)", mixed),
    ("Whole dataset (baseline)", np.ones(N, dtype=bool)),
]:
    n_subset = int(mask.sum())
    flan_in = int((flan_c & mask).sum())
    rate = flan_in / n_subset if n_subset > 0 else 0

    p_marginal = flan_c.mean()

    se = np.sqrt(p_marginal * (1 - p_marginal) / n_subset) if n_subset > 0 else 0
    z = (rate - p_marginal) / se if se > 0 else 0

    print(f"{label}")
    print(f"  Questions in subset: {n_subset}")
    print(f"  Flan correct: {flan_in}/{n_subset} = {rate*100:.1f}%")
    print(f"  Marginal Flan accuracy: {p_marginal*100:.1f}%")
    print(f"  Z (vs marginal): {z:+.2f}\n")

print("="*60)
print("INTERPRETATION GUIDE")
print("="*60)
print("If Flan on ALL_STRONG_WRONG ≈ 26.5% → correlated errors, no real reversal")
print("If Flan on ALL_STRONG_WRONG significantly > 26.5% → real expertise reversal")
print("If Flan on ALL_STRONG_WRONG significantly < 26.5% → strong-hard = Flan-hard too")

In [ ]:
import numpy as np
from scipy.stats import pearsonr

names = ['flan_t5', 'llama', 'qwen', 'gemma3n', 'deepseek']
M = np.stack([models_data[n] for n in names])

print("=== Pairwise correlation of correctness vectors ===\n")
print(f"{'':12s} " + " ".join([f"{n[:8]:>9s}" for n in names]))
for i, ni in enumerate(names):
    row = f"{ni[:8]:12s} "
    for j, nj in enumerate(names):
        r, _ = pearsonr(M[i], M[j])
        row += f"{r:>+9.3f} "
    print(row)

print("\n=== Phi coefficient (binary correlation, same as Pearson on 0/1) ===")
print("Interpretation: r > 0 means models tend to be right/wrong together")
print()

print("=== Conditional failure agreement P(B wrong | A wrong) ===\n")
print(f"{'A wrong | B wrong':22s} " + " ".join([f"{n[:8]:>9s}" for n in names]))
for i, ni in enumerate(names):
    row = f"{ni[:8] + ' wrong':22s} "
    n_a_wrong = (M[i] == 0).sum()
    for j, nj in enumerate(names):
        if i == j:
            row += f"{'—':>9s} "
            continue
        both_wrong = ((M[i] == 0) & (M[j] == 0)).sum()
        cond = both_wrong / n_a_wrong if n_a_wrong > 0 else 0

        marginal_b_wrong = (M[j] == 0).mean()
        row += f"{cond:>+9.3f} "
    print(row)

print(f"\nMarginal P(model wrong):")
for n in names:
    print(f"  {n:10s}: {(models_data[n] == 0).mean():.3f}")

print("\n=== Lift over independence (how much more likely B fails when A fails) ===")
print("Lift = P(B wrong | A wrong) / P(B wrong)")
print("Lift > 1 means correlated failures; lift = 1 means independent\n")
for i, ni in enumerate(names):
    for j, nj in enumerate(names):
        if i >= j:
            continue
        both_wrong = ((M[i] == 0) & (M[j] == 0)).sum()
        n_a_wrong = (M[i] == 0).sum()
        cond = both_wrong / n_a_wrong if n_a_wrong > 0 else 0
        marginal = (M[j] == 0).mean()
        lift = cond / marginal if marginal > 0 else float('inf')
        print(f"  {ni:8s} ↔ {nj:8s}: lift = {lift:.2f}x")

In [ ]:
import json
from datetime import datetime

summary = {
    'timestamp': datetime.now().isoformat(),
    'n_questions': N,
    'model_accuracies': {n: float(arr.mean()) for n, arr in models_data.items()},
    'k_intersection_test': {
        'k=1': {'observed': 94, 'null_mean': 119.9, 'z': -3.03},
        'k=2': {'observed': 54, 'null_mean': 118.1, 'z': -7.48},
        'k=3': {'observed': 43, 'null_mean': 48.5, 'z': -0.89},
        'k=4': {'observed': 31, 'null_mean': 6.7, 'z': +9.61},
    },
    'flan_stratified': {
        'all_strong_wrong': {'n': 143, 'flan_rate': 0.217, 'z_vs_marginal': -1.30},
        'all_strong_right': {'n': 383, 'flan_rate': 0.300, 'z_vs_marginal': +1.58},
        'mixed': {'n': 747, 'flan_rate': 0.256, 'z_vs_marginal': -0.56},
    },
    'pairwise_failure_lift': {
        'llama_qwen': 1.41, 'llama_gemma': 1.39, 'llama_deepseek': 1.51,
        'qwen_gemma': 1.61, 'qwen_deepseek': 1.74, 'gemma_deepseek': 1.67,
        'flan_llama': 1.02, 'flan_qwen': 1.07, 'flan_gemma': 1.06, 'flan_deepseek': 1.04,
    },
    'inter_classifier_reliability': {
        'kappa_5way': 0.181, 'gwet_ac1': 0.673, 'raw_agreement': 0.711,
    },
    'tested_and_null_hypotheses': [
        'Architecture-agnostic trap count (permutation test p=0.22)',
        'Trap questions bias-enriched vs random Llama errors (chi-sq p=0.20)',
        'High inter-classifier kappa (kappa=0.18, paradox-affected)',
        'Expertise reversal on all-strong-wrong subset (Flan z=-1.30)',
    ],
    'paper_framing': 'Correlated failure modes across LLM families in clinical reasoning',
    'next_steps': [
        'Bias-classify the 143 all-strong-wrong questions (~$0.50)',
        'MedMCQA replication of correlation pattern across all 5 models',
        'Draft paper outline',
    ]
}

with open('/content/drive/MyDrive/tonight_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Saved tonight_summary.json")
print(f"\nPaper framing: {summary['paper_framing']}")
print(f"\nTomorrow's first task: bias-classify the 143 all-strong-wrong questions.")
print("Reload this file when you start tomorrow to pick up cleanly.")

In [ ]:
import json, requests, time
import numpy as np
from collections import Counter

with open('/content/drive/MyDrive/llama_results.json') as f:
    llama = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/qwen_results.json') as f:
    qwen = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/gemma3n_results.json') as f:
    gemma = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/deepseek_results.json') as f:
    deepseek = {r['idx']: r for r in json.load(f)}
with open('/content/drive/MyDrive/flan_t5_results.json') as f:
    flan = {r['idx']: r for r in json.load(f)}

all_wrong_idx = []
for i in range(1273):
    if (llama[i]['correct'] == 0 and qwen[i]['correct'] == 0
        and gemma[i]['correct'] == 0 and deepseek[i]['correct'] == 0):
        all_wrong_idx.append(i)

print(f"All-strong-wrong questions: {len(all_wrong_idx)}")

print("\n=== Wrong-answer agreement analysis ===")

agreement_counts = Counter()
unanimous_idx = []
flan_correct_in_agree = []
flan_correct_in_unanimous = []

for i in all_wrong_idx:
    preds = [llama[i]['pred'], qwen[i]['pred'], gemma[i]['pred'], deepseek[i]['pred']]

    pred_counter = Counter(preds)
    most_common_pred, most_common_count = pred_counter.most_common(1)[0]
    agreement_counts[most_common_count] += 1
    if most_common_count == 4:
        unanimous_idx.append(i)
        if flan[i]['correct'] == 1:
            flan_correct_in_unanimous.append(i)
    if most_common_count >= 3:
        if flan[i]['correct'] == 1:
            flan_correct_in_agree.append(i)

print(f"\nDistribution of agreement on wrong answer (across 4 strong models):")
for k in sorted(agreement_counts.keys(), reverse=True):
    n = agreement_counts[k]
    pct = n / len(all_wrong_idx) * 100
    label = "UNANIMOUS — all 4 picked the same wrong answer" if k == 4 else f"{k} of 4 agreed on a wrong answer"
    print(f"  {k}/4: {n:>3} ({pct:.1f}%) — {label}")

print(f"\nFlan-correct rate in unanimous-wrong subset: "
      f"{len(flan_correct_in_unanimous)}/{len(unanimous_idx)} = "
      f"{len(flan_correct_in_unanimous)/max(1,len(unanimous_idx))*100:.1f}%")

print(f"\nUnanimous-wrong is the strongest possible shared-bias signal:")
print(f"  → {len(unanimous_idx)} questions where 4 different LLM families converged on the SAME wrong answer")
print(f"  → Under chance (3 distractors per question), 4 random wrong picks agreeing = (1/3)^3 ≈ 3.7%")
print(f"  → Observed unanimous rate: {len(unanimous_idx)/len(all_wrong_idx)*100:.1f}%")

print(f"\n\n=== Bias classification on the 143 shared-failure questions ===")
print(f"Using Llama-3.3-70B (same classifier as original 151 traps + 150 non-traps)")
print(f"Cost estimate: ~$0.35\n")

from datasets import load_dataset
from google.colab import userdata
dataset = load_dataset("GBaker/MedQA-USMLE-4-options")
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def classify_bias(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""
    try:
        r = requests.post(
            "https://api.together.xyz/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
                  "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": 10, "temperature": 0.0},
            timeout=30
        )
        if r.status_code != 200:
            return 'ERROR'
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS', 'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except:
        return 'ERROR'

shared_failure_labels = []
for i, idx in enumerate(all_wrong_idx):
    q = dataset['test'][idx]
    preds = [llama[idx]['pred'], qwen[idx]['pred'], gemma[idx]['pred'], deepseek[idx]['pred']]
    modal_wrong = Counter(preds).most_common(1)[0][0]

    label = classify_bias(
        question=q['question'],
        options=q['options'],
        gold=q['answer_idx'],
        wrong_answer=modal_wrong,
        api_key=TOGETHER_API_KEY
    )

    shared_failure_labels.append({
        'idx': idx,
        'bias_type': label,
        'modal_wrong': modal_wrong,
        'unanimous': len(set(preds)) == 1,
    })

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(all_wrong_idx)} done...")
    time.sleep(0.3)

with open('/content/drive/MyDrive/shared_failure_bias_labels.json', 'w') as f:
    json.dump(shared_failure_labels, f)

valid = [b for b in shared_failure_labels if b['bias_type'] != 'ERROR']
counts = Counter(b['bias_type'] for b in valid)
print(f"\n=== Shared-failure bias distribution (n={len(valid)}) ===")
for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/len(valid)*100:.1f}%)")

print(f"\n=== Comparison across three subsets ===")
print(f"{'Category':<22} {'Traps (n=151)':<15} {'Non-traps (n=150)':<18} {'Shared-fail (n=143)':<20}")
trap_dist = {'PREMATURE_CLOSURE': 118, 'ANCHORING_BIAS': 29, 'AVAILABILITY_BIAS': 3, 'OTHER': 1}
nontrap_dist = {'PREMATURE_CLOSURE': 107, 'ANCHORING_BIAS': 41, 'AVAILABILITY_BIAS': 2, 'OTHER': 0}
for cat in ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'OTHER']:
    t = trap_dist.get(cat, 0)
    nt = nontrap_dist.get(cat, 0)
    sf = counts.get(cat, 0)
    t_pct = t/151*100
    nt_pct = nt/150*100
    sf_pct = sf/max(1,len(valid))*100
    print(f"  {cat:<22} {t:>3} ({t_pct:>5.1f}%)   {nt:>3} ({nt_pct:>5.1f}%)        {sf:>3} ({sf_pct:>5.1f}%)")

In [ ]:
unanimous = [b for b in shared_failure_labels if b['unanimous']]
print(f"Unanimous-wrong questions: {len(unanimous)}\n")

modal_letters = Counter(b['modal_wrong'] for b in unanimous)
print("Which letter did all 4 models converge on?")
for letter in 'ABCD':
    print(f"  {letter}: {modal_letters.get(letter, 0)}")

gold_letters = Counter(dataset['test'][b['idx']]['answer_idx'] for b in unanimous)
print("\nGold answer letter distribution (for the 48):")
for letter in 'ABCD':
    print(f"  {letter}: {gold_letters.get(letter, 0)}")

print("\nBias classification on the unanimous-wrong 48:")
unanimous_bias = Counter(b['bias_type'] for b in unanimous)
for bias, count in sorted(unanimous_bias.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/len(unanimous)*100:.1f}%)")

print("\n" + "="*70)
print("FIVE EXAMPLE UNANIMOUS-WRONG QUESTIONS")
print("="*70)
for b in unanimous[:5]:
    q = dataset['test'][b['idx']]
    print(f"\n--- idx={b['idx']} ---")
    print(f"Question: {q['question'][:400]}{'...' if len(q['question'])>400 else ''}")
    print(f"\nOptions:")
    for letter, text in q['options'].items():
        marker = ""
        if letter == q['answer_idx']: marker = " ← GOLD"
        if letter == b['modal_wrong']: marker += " ← ALL 4 MODELS PICKED THIS"
        print(f"  {letter}: {text[:200]}{marker}")
    print(f"\nBias classification: {b['bias_type']}")
    print(f"Flan got it right: {flan[b['idx']]['correct'] == 1}")

In [ ]:
from scipy import stats
import numpy as np

trap_dist     = {'PREMATURE_CLOSURE': 118, 'ANCHORING_BIAS': 29, 'AVAILABILITY_BIAS': 3, 'OTHER': 1}
nontrap_dist  = {'PREMATURE_CLOSURE': 107, 'ANCHORING_BIAS': 41, 'AVAILABILITY_BIAS': 2, 'OTHER': 0}
shared_dist   = {'PREMATURE_CLOSURE': 95,  'ANCHORING_BIAS': 45, 'AVAILABILITY_BIAS': 2, 'OTHER': 1}
unanim_dist   = {'PREMATURE_CLOSURE': 29,  'ANCHORING_BIAS': 18, 'AVAILABILITY_BIAS': 0, 'OTHER': 1}

def pool(d):
    return [d['PREMATURE_CLOSURE'], d['ANCHORING_BIAS'],
            d.get('AVAILABILITY_BIAS',0) + d.get('OTHER',0)]

print("=== Pairwise chi-square: bias distribution differences ===\n")
comparisons = [
    ("Traps vs Shared-failure", pool(trap_dist), pool(shared_dist)),
    ("Traps vs Unanimous-wrong", pool(trap_dist), pool(unanim_dist)),
    ("Non-traps vs Shared-failure", pool(nontrap_dist), pool(shared_dist)),
    ("Non-traps vs Unanimous-wrong", pool(nontrap_dist), pool(unanim_dist)),
    ("Shared-failure vs Unanimous-wrong", pool(shared_dist), pool(unanim_dist)),
]

for label, a, b in comparisons:
    chi2, p, dof, _ = stats.chi2_contingency([a, b])
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {label:<40s}  chi2={chi2:.2f}  p={p:.4f}  {sig}")

print("\n=== Anchoring proportion across subsets (with 95% CI) ===\n")
def anchor_ci(d):
    n = sum(d.values())
    k = d['ANCHORING_BIAS']
    p = k / n
    se = np.sqrt(p * (1 - p) / n)
    return p, p - 1.96*se, p + 1.96*se, n

for label, d in [("Traps", trap_dist), ("Non-traps", nontrap_dist),
                 ("Shared-failure (143)", shared_dist), ("Unanimous-wrong (48)", unanim_dist)]:
    p, lo, hi, n = anchor_ci(d)
    print(f"  {label:<25s} n={n:<4d}  {p*100:>5.1f}%  95% CI [{lo*100:.1f}%, {hi*100:.1f}%]")

unanim_anchor_2x2 = np.array([
    [unanim_dist['ANCHORING_BIAS'], sum(unanim_dist.values()) - unanim_dist['ANCHORING_BIAS']],
    [trap_dist['ANCHORING_BIAS'], sum(trap_dist.values()) - trap_dist['ANCHORING_BIAS']]
])
or_val, p_fisher = stats.fisher_exact(unanim_anchor_2x2)
print(f"\nFisher exact (anchoring in unanimous-wrong vs traps):")
print(f"  Odds ratio: {or_val:.2f}")
print(f"  p-value:    {p_fisher:.4f}")

In [ ]:
import json
from datetime import datetime

summary_update = {
    'timestamp': datetime.now().isoformat(),
    'session': 'extended tonight session — positive results emerged',

    'primary_findings': {
        'unanimous_wrong_convergence': {
            'observed_rate': 0.336,
            'chance_rate': 0.037,
            'ratio_to_chance': 9.1,
            'n_unanimous': 48,
            'n_all_strong_wrong': 143,
            'note': 'Strongest finding — independent of bias classifier'
        },
        'pairwise_failure_lift': {
            'strong_strong_range': [1.39, 1.74],
            'flan_strong_range': [1.02, 1.07],
            'note': 'Capable models share blind spots; Flan independent'
        },
        'k_intersection_z_scores': {
            'k=1': -3.03, 'k=2': -7.48, 'k=3': -0.89, 'k=4': 9.61
        },
    },

    'secondary_findings': {
        'anchoring_elevation': {
            'trap_rate': 0.192,
            'shared_failure_rate': 0.315,
            'unanimous_wrong_rate': 0.375,
            'fisher_or': 2.52,
            'fisher_p': 0.012,
            'caveat': 'Depends on Llama-3.3-70B classifier; needs GPT-4o-mini triangulation'
        },
        'qualitative_examples': '5 unanimous-wrong questions inspected; all show convergence on USMLE-engineered novice distractors (e.g., DM+PAD → RAS instead of iliac aneurysm)'
    },

    'caveats_to_address_in_paper': [
        'Multiple comparisons across 5+ tests tonight — Bonferroni-corrected threshold ~0.01',
        'Anchoring elevation claim depends on Llama-3.3-70B labels; needs GPT-4o-mini triangulation',
        'n=48 unanimous-wrong is small; CI on anchoring rate is [24%, 51%]',
        'Inter-classifier kappa = 0.18 (paradox-affected; raw agreement 71%, AC1 = 0.67)',
        'Single dataset (MedQA-USMLE); MedMCQA replication needed for generalization'
    ],

    'paper_framing': 'Convergent Failure Modes Across LLM Families in Clinical Reasoning: Unanimous Distractor Convergence and Anchoring Bias Patterns',

    'tomorrow_priorities': [
        '1. GPT-4o-mini bias classification on the 143 shared-failure questions (triangulate anchoring elevation)',
        '2. Qualitative typology of all 48 unanimous-wrong questions (not just 5 examples)',
        '3. MedMCQA evaluation setup for all 5 models',
        '4. Begin paper outline'
    ]
}

with open('/content/drive/MyDrive/session_2_summary.json', 'w') as f:
    json.dump(summary_update, f, indent=2)

print("Saved session_2_summary.json")
print(f"\nPrimary finding: {summary_update['primary_findings']['unanimous_wrong_convergence']['ratio_to_chance']}× chance unanimous-wrong convergence")
print(f"Secondary finding: anchoring OR={summary_update['secondary_findings']['anchoring_elevation']['fisher_or']} p={summary_update['secondary_findings']['anchoring_elevation']['fisher_p']}")
print(f"\nCaveats: {len(summary_update['caveats_to_address_in_paper'])} documented")
print(f"\nTomorrow: triangulate anchoring claim with GPT-4o-mini, then MedMCQA")

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def classify_bias_gpt(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""
    try:
        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"model": "gpt-4o-mini",
                  "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": 10, "temperature": 0.0},
            timeout=30
        )
        if r.status_code != 200:
            return 'ERROR'
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS', 'FRAMING_EFFECT', 'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except:
        return 'ERROR'

print(f"Classifying the 143 shared-failure questions with GPT-4o-mini...")
print(f"Pairing each with prior Llama-3.3-70B label for direct comparison\n")

shared_failure_labels_gpt = []
for i, entry in enumerate(shared_failure_labels):
    idx = entry['idx']
    q = dataset['test'][idx]

    gpt_label = classify_bias_gpt(
        question=q['question'],
        options=q['options'],
        gold=q['answer_idx'],
        wrong_answer=entry['modal_wrong'],
        api_key=OPENAI_API_KEY
    )

    shared_failure_labels_gpt.append({
        'idx': idx,
        'bias_type_gpt': gpt_label,
        'bias_type_llama70b': entry['bias_type'],
        'modal_wrong': entry['modal_wrong'],
        'unanimous': entry['unanimous'],
    })

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(shared_failure_labels)} done...")
    time.sleep(0.2)

with open('/content/drive/MyDrive/shared_failure_bias_gpt4omini.json', 'w') as f:
    json.dump(shared_failure_labels_gpt, f)

valid = [b for b in shared_failure_labels_gpt if b['bias_type_gpt'] != 'ERROR']
print(f"\n=== Bias distribution on the 143 shared-failure questions ===\n")
print(f"{'Category':<22} {'Llama-3.3-70B':<18} {'GPT-4o-mini':<18}")
print("-" * 60)

llama_counts = Counter(b['bias_type_llama70b'] for b in valid)
gpt_counts = Counter(b['bias_type_gpt'] for b in valid)
total = len(valid)

for cat in ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'OTHER']:
    l = llama_counts.get(cat, 0)
    g = gpt_counts.get(cat, 0)
    print(f"  {cat:<22} {l:>3} ({l/total*100:>5.1f}%)     {g:>3} ({g/total*100:>5.1f}%)")

unanim_valid = [b for b in valid if b['unanimous']]
print(f"\n=== Bias distribution on the 48 UNANIMOUS-WRONG questions ===\n")
print(f"{'Category':<22} {'Llama-3.3-70B':<18} {'GPT-4o-mini':<18}")
print("-" * 60)

llama_unanim = Counter(b['bias_type_llama70b'] for b in unanim_valid)
gpt_unanim = Counter(b['bias_type_gpt'] for b in unanim_valid)
nu = len(unanim_valid)

for cat in ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'OTHER']:
    l = llama_unanim.get(cat, 0)
    g = gpt_unanim.get(cat, 0)
    print(f"  {cat:<22} {l:>3} ({l/nu*100:>5.1f}%)     {g:>3} ({g/nu*100:>5.1f}%)")

print(f"\n=== Anchoring rate across subsets — GPT-4o-mini perspective ===")

with open('/content/drive/MyDrive/bias_labels_gpt4omini.json') as f:
    earlier_gpt = json.load(f)
trap_idx_set = set(b['idx'] for b in earlier_gpt if b['is_trap'])
nontrap_idx_set = set(b['idx'] for b in earlier_gpt if not b['is_trap'])

gpt_trap_anchor = sum(1 for b in earlier_gpt if b['is_trap'] and b['bias_type'] == 'ANCHORING_BIAS')
gpt_nontrap_anchor = sum(1 for b in earlier_gpt if not b['is_trap'] and b['bias_type'] == 'ANCHORING_BIAS')
gpt_shared_anchor = gpt_counts.get('ANCHORING_BIAS', 0)
gpt_unanim_anchor = gpt_unanim.get('ANCHORING_BIAS', 0)

n_trap = len([b for b in earlier_gpt if b['is_trap']])
n_nontrap = len([b for b in earlier_gpt if not b['is_trap']])

print(f"  Traps              n={n_trap:<4d}  anchor={gpt_trap_anchor}  ({gpt_trap_anchor/n_trap*100:.1f}%)")
print(f"  Non-traps          n={n_nontrap:<4d}  anchor={gpt_nontrap_anchor}  ({gpt_nontrap_anchor/n_nontrap*100:.1f}%)")
print(f"  Shared-failure     n={total:<4d}  anchor={gpt_shared_anchor}  ({gpt_shared_anchor/total*100:.1f}%)")
print(f"  Unanimous-wrong    n={nu:<4d}  anchor={gpt_unanim_anchor}  ({gpt_unanim_anchor/nu*100:.1f}%)")

from scipy import stats
or_gpt, p_gpt = stats.fisher_exact([
    [gpt_unanim_anchor, nu - gpt_unanim_anchor],
    [gpt_trap_anchor, n_trap - gpt_trap_anchor]
])
print(f"\nFisher exact (GPT-4o-mini, unanimous-wrong vs traps):")
print(f"  Odds ratio: {or_gpt:.2f}")
print(f"  p-value:    {p_gpt:.4f}")

print(f"\n=== Decision ===")
print(f"Llama-3.3-70B: OR=2.52, p=0.012 (anchoring elevated in unanimous-wrong vs traps)")
print(f"GPT-4o-mini:   OR={or_gpt:.2f}, p={p_gpt:.4f}")
if p_gpt < 0.05 and or_gpt > 1.5:
    print("✓ Both classifiers agree: anchoring elevation is robust to classifier choice")
elif or_gpt > 1.5:
    print("~ Trend agrees, but GPT-4o-mini doesn't clear significance — directional support")
else:
    print("✗ GPT-4o-mini does NOT show the elevation — claim is classifier-dependent")

In [ ]:
import json
from datetime import datetime

final_update = {
    'timestamp': datetime.now().isoformat(),
    'session_end': 'GPT-4o-mini triangulation retracted secondary anchoring claim',

    'primary_findings_robust': {
        'unanimous_wrong_convergence': '48/143 = 33.6% vs 3.7% chance; 9.1x — classifier-independent',
        'pairwise_failure_lift': '1.39-1.74x for strong-strong; ~1.0x for Flan-strong',
        'k4_intersection_z': 9.61,
        'qualitative_examples': '4 LLM families converge on USMLE-engineered novice distractors',
    },

    'methodological_finding': {
        'pc_dominance_robust': 'Llama-3.3-70B: 66%, GPT-4o-mini: 84% — both classifiers agree PC dominates',
        'pc_vs_anchoring_unreliable': 'Llama labels 37.5% anchoring in unanimous-wrong; GPT labels 14.6%',
        'kappa': 0.181,
        'paper_implication': 'LLM-as-judge produces stable marginal distributions but unstable per-instance labels on semantically overlapping categories'
    },

    'retracted_tonight': {
        'anchoring_elevation_claim': 'OR=2.52 was Llama-3.3-70B artifact; GPT-4o-mini OR=0.95',
        'lesson': 'Always triangulate classifier-dependent claims before publishing'
    },

    'tomorrow_priorities': [
        '1. Qualitative typology of all 48 unanimous-wrong questions (read each, categorize the distractor pattern)',
        '2. MedMCQA evaluation setup — all 5 models on 5000 questions',
        '3. After MedMCQA: replicate unanimous-wrong analysis, see if pattern holds on second dataset',
        '4. Begin paper outline once MedMCQA confirms (or modifies) the convergence pattern'
    ],

    'paper_status': 'One primary finding + one methodological observation. Narrower than this morning but defensible at every reviewer challenge.'
}

with open('/content/drive/MyDrive/session_final.json', 'w') as f:
    json.dump(final_update, f, indent=2)

print("Saved session_final.json")
print(f"\nFinal status: Primary findings robust, secondary claim retracted, methodology paper viable")
print(f"\nGoodnight.")